# Desafío por equipos · Diseñá un experimento

**Consigna:** tu equipo diseña un **experimento** para su aplicación y lo escribe en un **canvas**. **No hace falta entrenar un modelo grande.** El producto es una decisión bien pensada: qué línea base hay que superar, qué arquitectura elegís y por qué, cómo dividís los datos, con qué métrica, cuánto cuesta, quién puede salir perjudicado y **cuándo descartarías la red**.

**Esta actividad es de pensar, no de codear.** Escribís palabras en un diccionario. Solo los equipos 1 y 6, que tienen datos tabulares, pueden correr una **comparación chica** para probar el método; el resto trabaja directamente en el canvas. El código te devuelve **preguntas**, no respuestas.

| Minutos | Qué hacen |
| --- | --- |
| 0–5 | Leer la tarjeta, asignar los roles y discutir las preguntas. |
| 5–12 | Línea base y arquitectura. **A los 12 minutos, control obligatorio.** |
| 12–20 | Split, criterio de detención, métrica, costo y revisión ética. |
| 20–25 | Condición de abandono, revisión y ensayo de la defensa de 3 minutos. |

**Roles:** quien conduce, quien completa el canvas, quien valida y quien presenta.

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería, la instala en el mismo Python que ejecuta este notebook; si ya está todo, no hace nada.

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "matplotlib": "matplotlib", "jinja2": "jinja2"}
INSTALAR_XGBOOST = True          # opcional: ponelo en True para sumar XGBoost a la comparación de modelos
if INSTALAR_XGBOOST: REQUISITOS["xgboost"] = "xgboost"

print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")


## 0 · Preparación (no hace falta tocarla)

Esta celda carga las cuatro aplicaciones, genera los datos sintéticos de NovaRetail (caso 1 y caso 2) y define las funciones. Si no la corrés, ninguna de las siguientes funciona.

In [ ]:
CAMPOS = ['linea_base', 'arquitectura', 'por_que_arquitectura', 'split', 'criterio_detencion', 'metrica', 'costo', 'revision_etica', 'riesgo_principal', 'condicion_abandono']

DESAFIOS = {1: {'titulo': 'Abandono de clientes',
     'modalidad': 'Tabular',
     'situacion': 'NovaRetail quiere decidir a quién contactar antes de que abandone. Hay 8.000 clientes y 21 variables; abandona el 14,3 %. '
                  'Contactar de más cuesta 5 y perder a un cliente, 60.',
     'complicacion': 'Pocos casos positivos y un error cuesta mucho más que el otro. La regresión logística de la clase 2 ya llega a un AUC de '
                     '0,837.',
     'pedido': 'Diseñá el experimento que decida si una red justifica su costo frente a esa línea base.',
     'cuidado': 'Una red que gana por 0,005 de AUC, ¿vale lo que cuesta?',
     'preguntas': ['¿Cuánto tiene que ganar la red para justificar su costo?',
                   '¿Cómo sabés que la ganancia no es suerte de una semilla?',
                   '¿Qué métrica refleja que un error cuesta más que el otro?'],
     'claves': [('metrica', 'costo|recall|precisi|umbral|F1', 'El error no cuesta lo mismo para los dos lados: ¿tu métrica lo refleja?'),
                ('condicion_abandono', '\\d|semilla|ruido|supera|gana', '¿Cuánto tiene que ganar la red para que no la descartes?')],
     'codigo': True},
 2: {'titulo': 'Pronóstico de demanda semanal',
     'modalidad': 'Secuencia',
     'situacion': 'NovaRetail repone 400 productos cada semana. Hay 3 años de ventas semanales y se quiere pronosticar la semana siguiente de cada '
                  'producto.',
     'complicacion': 'El orden del tiempo manda: mezclar semanas filtra el futuro. En una serie de ejemplo, el error es 8,7 con división al azar y '
                     '15,8 con división temporal.',
     'pedido': 'Diseñá el experimento: cómo se divide, contra qué se compara y cuándo una red recurrente justificaría su costo.',
     'cuidado': 'Un resultado que mejora al mezclar el tiempo es una fuga, no una mejora.',
     'preguntas': ['¿Cuál es la línea base más simple que habría que superar?',
                   '¿Cómo dividís los datos sin mirar el futuro?',
                   '¿Qué error cuesta más: el faltante o el sobrante?'],
     'claves': [('split', 'tiempo|tempor|cronol|fecha|semanas? pasada|últim', 'Tu serie tiene orden: ¿cómo dividís sin mirar el futuro?'),
                ('metrica', 'error|MAE|MAPE|RMSE|desv', 'Estás pronosticando una cantidad: ¿con qué error la medís?')],
     'codigo': False},
 3: {'titulo': 'Reclamos por texto',
     'modalidad': 'Texto',
     'situacion': 'Atención al cliente recibe miles de reclamos escritos y los deriva a mano. Hay 6.000 reclamos etiquetados con 5 motivos.',
     'complicacion': 'Hay pocos reclamos etiquetados por motivo y el lenguaje cambia entre regiones.',
     'pedido': 'Diseñá el experimento: ¿se reutiliza un modelo preentrenado o se entrena uno propio? ¿Contra qué línea base?',
     'cuidado': 'Un modelo que anda bien en una región puede fallar en otra: ¿cómo lo verificás?',
     'preguntas': ['¿Cuántos datos etiquetados necesitás para entrenar uno propio?',
                   '¿Qué línea base con palabras ya resuelve buena parte?',
                   '¿Cómo medís el desempeño región por región?'],
     'claves': [('por_que_arquitectura',
                 'preentrenad|reutiliz|ajust|pocos datos|etiquet',
                 '¿Por qué reutilizar o entrenar? Pensá en cuántos reclamos etiquetados tenés.'),
                ('revision_etica',
                 'región|regional|lenguaje|dialecto|sesgo|grupo',
                 'El lenguaje cambia entre regiones: ¿quién podría recibir peor servicio?')],
     'codigo': False},
 4: {'titulo': 'Productos devueltos por imagen',
     'modalidad': 'Imagen',
     'situacion': 'El área de devoluciones decide si aprobar una devolución mirando una foto. Hay 3.000 fotos etiquetadas; 1 de cada 8 muestra un '
                  'daño.',
     'complicacion': 'Hay pocas imágenes de producto dañado y hace falta explicar cada decisión al cliente.',
     'pedido': 'Diseñá el experimento: ¿qué arquitectura, con qué datos y cómo se explica una decisión?',
     'cuidado': 'Aprobar un producto dañado cuesta más que revisarlo a mano: ¿tu métrica lo refleja?',
     'preguntas': ['¿Qué hacés con tan pocas imágenes dañadas?',
                   '¿Qué reglas simples sobre la ficha del producto habría que superar?',
                   '¿Cómo le explicás una decisión a un cliente?'],
     'claves': [('metrica', 'recall|precisi|falsos|daño|costo', 'Un error cuesta más que el otro: ¿qué métrica lo refleja?'),
                ('riesgo_principal', 'pocas|pocos|desbalance|raro|explic', '¿Qué problema trae tener pocas imágenes de lo que querés detectar?')],
     'codigo': False},
 5: {'titulo': 'Recomendación de productos',
     'modalidad': 'Secuencia',
     'situacion': 'NovaRetail quiere sugerir qué comprar a continuación a partir del historial de cada cliente.',
     'complicacion': 'Dominan los productos más vendidos y los clientes nuevos no tienen historial.',
     'pedido': 'Diseñá el experimento: ¿cómo se mide que la recomendación mejora a «recomendar los más vendidos»?',
     'cuidado': 'Un modelo que recomienda siempre lo más vendido acierta mucho y no aporta nada.',
     'preguntas': ['¿Qué métrica premia los aciertos que no son los más vendidos?',
                   '¿Qué hacés con un cliente sin historial?',
                   '¿Cómo dividís el tiempo?'],
     'claves': [('linea_base', 'más vendid|popular|top|ranking', '¿Cuál es la línea base más simple para recomendar?'),
                ('metrica', 'top|ranking|acierto|MAP|recall|cobertura|diversidad|primeras', '¿Qué métrica mide si lo recomendado se compra?')],
     'codigo': False},
 6: {'titulo': 'Operaciones sospechosas',
     'modalidad': 'Tabular y secuencia',
     'situacion': 'Una auditoría revisa operaciones sospechosas con un cupo diario. Hay 100.000 operaciones y solo 0,5 % fue confirmada como '
                  'sospechosa.',
     'complicacion': 'Muy pocos casos positivos, un error cuesta mucho más que el otro y el comportamiento cambia con el tiempo.',
     'pedido': 'Diseñá el experimento: ¿cómo se valida con tan pocos casos y cuándo se descarta la red?',
     'cuidado': 'Con 0,5 % de casos, decir que nada es sospechoso acierta el 99,5 %.',
     'preguntas': ['¿Qué métrica sirve cuando casi todo es normal?',
                   '¿Cómo se valida con tan pocos positivos?',
                   '¿Qué pasa cuando el comportamiento cambia?'],
     'claves': [('metrica', 'recall|precisi|costo|cupo|top|mayor riesgo|F1', 'Casi todo es normal: ¿tu métrica mira lo raro?'),
                ('condicion_abandono', '\\d|semilla|supera|cupo', '¿Qué resultado concreto te haría descartar la red?')],
     'codigo': True},
 7: {'titulo': 'Asistente generativo de atención',
     'modalidad': 'Generativo',
     'situacion': 'Se evalúa un asistente que redacte respuestas a consultas de clientes con apoyo de las preguntas frecuentes y el historial.',
     'complicacion': 'El modelo puede inventar respuestas, el historial tiene datos sensibles y cada consulta tiene un costo.',
     'pedido': 'Diseñá el experimento: ¿se justifica un modelo generativo frente a un buscador de preguntas frecuentes?',
     'cuidado': 'Un asistente que suena seguro y se equivoca puede ser peor que no tener asistente.',
     'preguntas': ['¿Cómo medís que una respuesta es correcta?',
                   '¿Qué datos nunca pueden llegar al modelo?',
                   '¿Cuánto cuesta cada consulta contra el buscador?'],
     'claves': [('linea_base',
                 'buscador|preguntas frecuentes|FAQ|respuesta oficial',
                 '¿Qué solución simple con las preguntas frecuentes habría que superar?'),
                ('revision_etica',
                 'sensible|personal|inventa|alucin|privacidad|dato',
                 '¿Qué datos sensibles manejás y qué pasa si el modelo inventa?')],
     'codigo': False}}

COLUMNAS = ["id_cliente", "fecha_alta", "canal", "segmento", "region", "edad", "antiguedad_meses", "compras_12m",
            "ticket_promedio", "dias_desde_ultima_compra", "contactos_soporte_90d", "usa_app", "descuento_promedio",
            "satisfaccion", "motivo_baja", "abandono_90d"]


def generar_novaretail(n=8000, semilla=2025, duplicados=60):
    rng = np.random.default_rng(semilla)
    fecha_corte = pd.Timestamp("2025-12-31")
    canal = rng.choice(["web", "app", "tienda"], size=n, p=[0.40, 0.25, 0.35])
    # composición: web concentra más clientes del segmento Esencial (esto confunde la lectura por canal)
    p_seg = {"web": [0.62, 0.28, 0.10], "app": [0.30, 0.45, 0.25], "tienda": [0.28, 0.42, 0.30]}
    segmento = np.array([rng.choice(["Esencial", "Plus", "Premium"], p=p_seg[c]) for c in canal])
    region = rng.choice(["Centro", "Norte", "Sur", "Litoral", "Cuyo"], size=n, p=[0.30, 0.20, 0.18, 0.17, 0.15])
    edad = np.clip(rng.normal(41, 12, n), 18, 80).round().astype(int)
    alta = pd.Timestamp("2021-01-01") + pd.to_timedelta(rng.integers(0, 1730, n), unit="D")
    antig = ((fecha_corte - alta).days / 30.4).round().astype(int)
    base_compras = np.select([segmento == "Esencial", segmento == "Plus"], [5, 9], 14)
    compras = rng.poisson(base_compras)
    ticket = np.round(np.select([segmento == "Esencial", segmento == "Plus"], [35, 62], 110) * rng.lognormal(0, 0.25, n), 2)
    dias = np.clip(rng.gamma(2.2, 24, n), 1, 365).round().astype(int)
    soporte = rng.poisson(np.where(segmento == "Esencial", 1.2, 0.6))
    usa_app = np.where(canal == "app", 1, (rng.random(n) < 0.35).astype(int))
    descuento = np.clip(rng.normal(np.where(segmento == "Esencial", 0.14, 0.08), 0.05, n), 0, 0.35).round(3)
    sat_real = np.clip(np.round(rng.normal(3.7 - 0.12 * soporte + 0.15 * (segmento == "Premium"), 0.9, n)), 1, 5)

    z_dias = (dias - 60) / 40
    logit = (-2.45
             + 0.85 * (segmento == "Esencial") - 0.45 * (segmento == "Premium")
             + 0.80 * z_dias + 0.30 * soporte - 0.42 * (sat_real - 3.7)
             - 0.50 * usa_app - 0.012 * (antig - 24) - 0.04 * (compras - 8)
             + 0.55 * ((canal == "tienda") & (segmento == "Esencial")))
    p = 1 / (1 + np.exp(-logit))
    abandono = (rng.random(n) < p).astype(int)

    # faltantes: la satisfacción falta más entre quienes abandonan (faltante informativo)
    sat = sat_real.copy()
    sat[rng.random(n) < np.where(abandono == 1, 0.30, 0.09)] = np.nan
    ticket = ticket.astype(float)
    ticket[rng.random(n) < 0.03] = np.nan
    region_s = pd.Series(region, dtype=object)
    region_s[rng.random(n) < 0.02] = np.nan
    # valores imposibles
    edad = edad.astype(float)
    malos = rng.random(n) < 0.004
    edad[malos] = rng.choice([0, 150, -1], size=malos.sum())
    # fuga: el motivo de baja solo existe para quienes ya abandonaron
    motivo = np.where(abandono == 1, rng.choice(["precio", "servicio", "competencia", "mudanza"], size=n), None)

    df = pd.DataFrame({
        "id_cliente": [f"NR{i:06d}" for i in range(1, n + 1)],
        "fecha_alta": alta.strftime("%Y-%m-%d"),
        "canal": canal, "segmento": segmento, "region": region_s,
        "edad": edad, "antiguedad_meses": antig, "compras_12m": compras,
        "ticket_promedio": ticket, "dias_desde_ultima_compra": dias,
        "contactos_soporte_90d": soporte, "usa_app": usa_app,
        "descuento_promedio": descuento, "satisfaccion": sat,
        "motivo_baja": motivo, "abandono_90d": abandono,
    })
    # duplicados: filas repetidas por un error de carga
    if duplicados:
        df = pd.concat([df, df.sample(duplicados, random_state=semilla)], ignore_index=True)
    return df.sample(frac=1, random_state=semilla).reset_index(drop=True)


import re, time
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
try:
    from xgboost import XGBClassifier
    def modelo_arboles(semilla): return XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, random_state=semilla, n_jobs=1, eval_metric="logloss")
    NOMBRE_ARBOLES = "XGBoost (100 árboles, prof. 3)"
except Exception:
    from sklearn.ensemble import HistGradientBoostingClassifier
    def modelo_arboles(semilla): return HistGradientBoostingClassifier(max_depth=3, random_state=semilla)
    NOMBRE_ARBOLES = "Boosting de scikit-learn (XGBoost no está instalado)"

NUM = ["edad", "antiguedad_meses", "compras_12m", "ticket_promedio", "dias_desde_ultima_compra", "contactos_soporte_90d", "usa_app", "descuento_promedio", "satisfaccion", "satisfaccion_faltante"]
NUEVAS = ["ciclos_sin_comprar", "gasto_anual", "contactos_por_compra"]
CAT = ["canal", "segmento", "region"]

def preparar(caso=1):
    """Caso 1: NovaRetail tal cual. Caso 2: los mismos clientes, tres variables nuevas y otra regla de abandono (umbral, interacción y efectos que no son una línea)."""
    df = generar_novaretail().drop_duplicates("id_cliente").reset_index(drop=True)
    df.loc[(df.edad < 18) | (df.edad > 100), "edad"] = np.nan
    df["satisfaccion_faltante"] = df.satisfaccion.isna().astype(int)
    c = df.compras_12m
    df["ciclos_sin_comprar"] = df.dias_desde_ultima_compra * np.maximum(c, 1) / 365
    df["gasto_anual"] = np.where(c == 0, 0.0, df.ticket_promedio * c)
    df["contactos_por_compra"] = df.contactos_soporte_90d / (c + 1)
    if caso == 2:
        azar = np.random.default_rng(2026)                       # flujo aparte: el caso 1 no cambia
        gasto = df.gasto_anual.fillna(df.gasto_anual.median())
        z = (-2.8 + 0.55 * (df.segmento == "Esencial") - 0.35 * (df.segmento == "Premium") + 0.35 * (df.dias_desde_ultima_compra - 60) / 40
             + 2.6 * (df.ciclos_sin_comprar > 2.0) + 2.4 * ((df.usa_app == 0) & (df.contactos_soporte_90d >= 3))
             + 1.6 * ((df.edad < 25) | (df.edad > 65)) - 1.2 * (gasto > 600))
        df["abandono_90d"] = (azar.random(len(df)) < 1 / (1 + np.exp(-z))).astype(int)
    return df

def comparar(caso=1, prevalencia=None, semillas=5, con_nuevas=None):
    """Compara regresión, red densa pequeña y árboles con la misma partición y la misma métrica.
    caso: 1 o 2 · prevalencia: por ejemplo 0.005 para dejar solo 0,5 % de casos positivos · con_nuevas: por defecto, False en el caso 1 y True en el caso 2."""
    df = preparar(caso)
    if con_nuevas is None: con_nuevas = (caso == 2)
    if prevalencia:
        neg, pos = df[df.abandono_90d == 0], df[df.abandono_90d == 1]
        n_pos = int(round(prevalencia * len(neg) / (1 - prevalencia))); df = pd.concat([neg, pos.sample(n_pos, random_state=0)]).reset_index(drop=True)
    num = NUM + (NUEVAS if con_nuevas else [])
    X, y = df[num + CAT], df["abandono_90d"]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
    def armar(m): return Pipeline([("prep", ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), num),
                                                                ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), CAT)])), ("modelo", m)])
    modelos = [("Regresión logística (L2)", lambda s: LogisticRegression(C=1.0, max_iter=3000), 1),
               ("Red densa (64, 32)", lambda s: MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True, validation_fraction=0.15, n_iter_no_change=10, max_iter=500, random_state=s), semillas),
               (NOMBRE_ARBOLES, modelo_arboles, semillas)]
    filas = []
    for nombre, fab, reps in modelos:
        aucs, tops, tiempos = [], [], []
        for s in range(reps):
            t0 = time.time(); m = armar(fab(s)).fit(Xtr, ytr); tiempos.append(time.time() - t0)
            p = m.predict_proba(Xte)[:, 1]; aucs.append(roc_auc_score(yte, p))
            n_top = max(1, int(round(0.05 * len(yte)))); tops.append(np.asarray(yte)[np.argsort(-p)[:n_top]].mean())
        filas.append({"modelo": nombre, "AUC (media)": round(np.mean(aucs), 3), "AUC mín–máx": f"{min(aucs):.3f}–{max(aucs):.3f}", "aciertos en el 5 % de mayor riesgo": f"{100 * np.mean(tops):.0f} %", "entrena en (s)": round(np.mean(tiempos), 2)})
    tasa = f"{100 * y.mean():.1f}".replace(".", ","); n_clientes = f"{len(df):,}".replace(",", ".")
    print(f"Caso {caso} · {n_clientes} clientes · tasa de abandono {tasa} % · casos positivos en la prueba: {int(np.sum(yte))} · variables nuevas: {'sí' if con_nuevas else 'no'}")
    return pd.DataFrame(filas).set_index("modelo")

CRITERIOS = ["Línea base explícita", "Pregunta definida", "Split sin fuga", "Métrica con sentido", "Costo declarado", "Riesgo nombrado", "Condición de abandono"]
def mostrar_equipo(equipo):
    d = DESAFIOS[equipo]
    print(f"EQUIPO {equipo} · {d['titulo']} ({d['modalidad']})\n" + "=" * 60)
    for et, k in [("Situación", "situacion"), ("Complicación", "complicacion"), ("Pedido", "pedido"), ("Cuidado con", "cuidado")]: print(f"\n{et}: {d[k]}")
    print("\nPreguntas para pensar antes de escribir:"); [print(f"  {i}. {q}") for i, q in enumerate(d["preguntas"], 1)]

def revisar_canvas(canvas, equipo):
    d = DESAFIOS[equipo]; sin = {c: str(canvas.get(c, "")).strip() for c in CAMPOS}
    vacios = [c for c, v in sin.items() if not v]
    if vacios:
        print("Todavía faltan campos: " + ", ".join(vacios) + ".\nCompletalos y volvé a correr esta celda."); return None
    pistas = {c: [] for c in CRITERIOS}
    if len(sin["linea_base"]) < 15: pistas["Línea base explícita"].append("¿Qué modelo o regla simple habría que superar? Escribilo con más detalle.")
    if len(sin["arquitectura"]) < 8 or len(sin["por_que_arquitectura"]) < 40: pistas["Pregunta definida"].append("Decí qué arquitectura elegís y por qué es proporcional al problema, en una o dos frases.")
    if len(sin["split"]) < 20: pistas["Split sin fuga"].append("Contá cómo dividís los datos y cómo evitás que el modelo vea algo que no debería.")
    if len(sin["metrica"]) < 15: pistas["Métrica con sentido"].append("¿Por qué esa métrica sirve para el costo del error de tu problema?")
    if len(sin["costo"]) < 15: pistas["Costo declarado"].append("¿Cuánto cuesta entrenar y operar? Hablá de tiempo, datos y personas.")
    if len(sin["revision_etica"]) < 15 or len(sin["riesgo_principal"]) < 10: pistas["Riesgo nombrado"].append("¿Quién podría salir perjudicado y cuál es el riesgo principal?")
    if len(sin["criterio_detencion"]) < 15: pistas["Split sin fuga"].append("¿Cuándo se corta el entrenamiento? Nombrá qué mirás para decidirlo.")
    if len(sin["condicion_abandono"]) < 15: pistas["Condición de abandono"].append("¿Qué resultado concreto te haría descartar la red?")
    destino = {"linea_base": "Línea base explícita", "por_que_arquitectura": "Pregunta definida", "split": "Split sin fuga", "metrica": "Métrica con sentido", "costo": "Costo declarado", "revision_etica": "Riesgo nombrado", "riesgo_principal": "Riesgo nombrado", "condicion_abandono": "Condición de abandono"}
    for campo, patron, pista in d["claves"]:
        if not re.search(patron, sin[campo], re.I): pistas[destino[campo]].append(pista)
    print(f"REVISIÓN · equipo {equipo}: criterios de calidad, con preguntas y no respuestas\n" + "=" * 60)
    cumplidos = 0
    for i, c in enumerate(CRITERIOS, 1):
        if pistas[c]:
            print(f"\n{i}. {c}"); [print("   → Pista:", x) for x in pistas[c]]
        else: print(f"\n{i}. {c}  ✔"); cumplidos += 1
    print(f"\nCriterios sin pistas: {cumplidos} de 7")
    return cumplidos

def mostrar_canvas(canvas, equipo):
    d = DESAFIOS[equipo]; nombres = {"linea_base": "Línea base", "arquitectura": "Arquitectura", "por_que_arquitectura": "Por qué esa arquitectura", "split": "Split", "criterio_detencion": "Criterio de detención", "metrica": "Métrica", "costo": "Costo", "revision_etica": "Revisión ética", "riesgo_principal": "Riesgo principal", "condicion_abandono": "Condición de abandono"}
    print(f"CANVAS DE EXPERIMENTO · equipo {equipo} · {d['titulo']}\n" + "=" * 60)
    for c in CAMPOS: print(f"\n{nombres[c]}:\n  {canvas.get(c, '')}")


def control_12(canvas):
    print("CONTROL A LOS 12 MINUTOS: mostrale esto al facilitador\n" + "-" * 60)
    for campo, nombre in [("linea_base", "Línea base"), ("arquitectura", "Arquitectura")]:
        valor = str(canvas.get(campo, "")).strip()
        print(("✔ " if valor else "… ") + f"{nombre:<14} {valor or 'pendiente'}")

## 1 · Tu equipo

Cambiá el número por el de tu equipo (del 1 al 7) y corré la celda.

In [ ]:
MI_EQUIPO = 1          # ← cambialo por el número de tu equipo

mostrar_equipo(MI_EQUIPO)

## 2 · Pensá antes de escribir

Conversen **cinco minutos sin escribir nada**. Estas cinco preguntas son las que el facilitador va a devolverles:

1. **¿Qué decisión mejora?** ¿Qué se hace distinto con el resultado?
2. **¿Para qué población?** ¿Quién entra y quién queda afuera?
3. **¿Hay fuga o sesgo?** ¿Qué sabe el modelo que no debería saber?
4. **¿Cómo se valida?** ¿Qué partición, qué métrica y cuántas semillas?
5. **¿Cuándo se abandona?** ¿Qué resultado haría descartar la red?

## 3 · Tu canvas

Completá cada campo **entre comillas**, con tus palabras. Una o dos frases por campo alcanzan.

- `linea_base`: el modelo o la regla simple que hay que superar.
- `arquitectura` y `por_que_arquitectura`: qué elegís y por qué es proporcional al problema.
- `split`: cómo dividís los datos y cómo evitás la fuga y el sesgo.
- `criterio_detencion`: cuándo se corta el entrenamiento.
- `metrica`: con qué medís, y por qué sirve para el costo del error.
- `costo`: lo que cuesta entrenar y operar.
- `revision_etica`: quién podría salir perjudicado.
- `riesgo_principal`: el riesgo que más te preocupa.
- `condicion_abandono`: qué resultado concreto te haría descartar la red.

In [ ]:
canvas = {
    "linea_base": "",
    "arquitectura": "",
    "por_que_arquitectura": "",
    "split": "",
    "criterio_detencion": "",
    "metrica": "",
    "costo": "",
    "revision_etica": "",
    "riesgo_principal": "",
    "condicion_abandono": "",
}

## Control a los 12 minutos

Mostrale este resultado al facilitador antes de seguir con el resto del canvas.

In [ ]:
control_12(canvas)

## 4 · Opcional: un ensayo con datos (equipos 1 y 6)

Los equipos con datos tabulares pueden probar el método con NovaRetail. **Los demás pueden saltear esta sección.** Cada comparación tarda unos segundos: usa la misma partición y la misma métrica para tres modelos, con cinco semillas.

**Equipo 1.** Corré los dos casos y respondé: ¿en cuál gana la red? ¿Qué cambió entre uno y otro?

In [ ]:
comparar(1)

In [ ]:
comparar(2)

In [ ]:
interpretacion_1 = ""      # ← solo equipo 1: ¿qué cambió entre el caso 1 y el caso 2?
print(interpretacion_1)

**Equipo 6.** Dejá solo una parte de los casos positivos y mirá qué pasa. Fijate en **cuántos casos positivos tiene la prueba** y en lo que le pasa a cada modelo.

In [ ]:
comparar(1, prevalencia=0.02)

In [ ]:
comparar(1, prevalencia=0.005)

In [ ]:
interpretacion_6 = ""      # ← solo equipo 6: ¿qué le pasa a la red con muy pocos positivos? ¿Qué decisión tomás?
print(interpretacion_6)

## 5 · Revisión: preguntas, no respuestas

Corré la revisión cuando hayan completado todo el canvas. Si hay pistas, **discutan primero** qué supuesto cuestiona cada una, y después ajusten el canvas.

In [ ]:
revisar_canvas(canvas, MI_EQUIPO)

## 6 · Para presentar

En la defensa tienen **3 minutos** para presentar y 2 para responder preguntas.

In [ ]:
mostrar_canvas(canvas, MI_EQUIPO)

## 7 · Revisión cruzada

Revisá el canvas de otro equipo con la lista de **siete criterios**: línea base explícita, pregunta definida, split sin fuga, métrica con sentido, costo declarado, riesgo nombrado y condición de abandono. Anotá una devolución escrita:

In [ ]:
devolucion = ""      # ← una frase por criterio que no esté cumplido
print(devolucion)